Import libraries

In [2]:
import pandas as pd
import numpy as np
import requests
from functools import reduce

Helpers

In [3]:
def make_bbl(boro, block, lot):
    """
    NYC BBL = borough(1) + block(5) + lot(4)
    """
    return (
        boro.astype(str).str.extract(r"(\d+)")[0].str.zfill(1) +
        block.astype(str).str.extract(r"(\d+)")[0].str.zfill(5) +
        lot.astype(str).str.extract(r"(\d+)")[0].str.zfill(4)
    )

In [4]:
def socrata_get(dataset_id, select="*", where=None, limit=50000, max_rows=None):
    """
    Pull Socrata data in chunks to avoid direct import limits.
    """
    base = f"https://data.cityofnewyork.us/resource/{dataset_id}.json"
    frames = []
    offset = 0

    while True:
        params = {
            "$select": select,
            "$limit": limit,
            "$offset": offset
        }
        if where:
            params["$where"] = where

        r = requests.get(base, params=params)
        r.raise_for_status()
        data = r.json()

        if not data:
            break

        frames.append(pd.DataFrame(data))
        offset += limit

        if max_rows and offset >= max_rows:
            break

        print(f"Pulled {offset:,} rows from {dataset_id}...")

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

Data pulls

1. Lien Sale Eligibility List

In [5]:
lien = socrata_get(
    dataset_id="9rz4-mjek",
    select="*"
)

lien.columns = (
    lien.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

print(lien.shape)
display(lien.head())
display(lien["cycle"].value_counts(dropna=False))

Pulled 50,000 rows from 9rz4-mjek...
Pulled 100,000 rows from 9rz4-mjek...
Pulled 150,000 rows from 9rz4-mjek...
Pulled 200,000 rows from 9rz4-mjek...
Pulled 250,000 rows from 9rz4-mjek...
Pulled 300,000 rows from 9rz4-mjek...
(264142, 13)


,month,cycle,borough,block,lot,tax_class_code,building_class,community_board,council_district,house_number,street_name,zip_code,water_debt_only
0,2019-04-17T00:00:00.000,90 Day Notice,1,16,3,4,Z9,101,1,401,SOUTH END AVENUE,10280,NO
1,2019-04-17T00:00:00.000,90 Day Notice,1,17,1213,4,RB,101,NaN,50,WEST STREET,10006,NO
2,2019-04-17T00:00:00.000,90 Day Notice,1,17,1215,4,RB,101,NaN,50,WEST STREET,10006,NO
3,2019-04-17T00:00:00.000,90 Day Notice,1,18,1073,2,R4,101,1,88,GREENWICH STREET,10006,NO
4,2019-04-17T00:00:00.000,90 Day Notice,1,18,1270,2,R4,101,1,88,GREENWICH STREET,10006,NO


cycle
90 Day Notice     63185
60 Day Notice     53767
30 Day Notice     44726
10 Day Notice     37942
90 Days Notice    18907
60 Days Notice    15718
Final Sale        11107
30 Days Notice     9678
10 Days Notice     9112
Name: count, dtype: int64

In [6]:
lien_all = lien.copy()
final_sale = lien[lien["cycle"] == "Final Sale"].copy()

In [7]:
final_sale["boro_code"] = (
    final_sale["borough"]
    .astype(str)
    .str.upper()
    .str.strip()
    .replace({
        "MANHATTAN": "1",
        "BRONX": "2",
        "BROOKLYN": "3",
        "QUEENS": "4",
        "STATEN ISLAND": "5"
    })
)

final_sale["bbl"] = make_bbl(
    final_sale["boro_code"],
    final_sale["block"],
    final_sale["lot"]
)

final_sale_property = (
    final_sale
    .drop_duplicates("bbl")
    .copy()
)

print(final_sale_property.shape)
display(final_sale_property[["month", "cycle", "borough", "block", "lot", "bbl"]].head())

(9763, 15)


,month,cycle,borough,block,lot,bbl
65761,2019-10-01T00:00:00.000,Final Sale,1,26,1260,1000261260
65762,2019-10-01T00:00:00.000,Final Sale,1,26,1273,1000261273
65763,2019-10-01T00:00:00.000,Final Sale,1,27,1040,1000271040
65764,2019-10-01T00:00:00.000,Final Sale,1,31,1068,1000311068
65765,2019-10-01T00:00:00.000,Final Sale,1,44,1265,1000441265


2. Primary Land Use Tax Lot Output

In [8]:
pluto_cols = [
    "borough",
    "block",
    "lot",
    "bbl",
    "cd",
    "ct2010",
    "cb2010",
    "zipcode",
    "landuse",
    "bldgclass",
    "ownertype",
    "lotarea",
    "bldgarea",
    "resarea",
    "unitsres",
    "unitstotal",
    "yearbuilt",
    "latitude",
    "longitude"
]

pluto = socrata_get(
    dataset_id="64uk-42ks",
    select=", ".join(pluto_cols)
)

Pulled 50,000 rows from 64uk-42ks...
Pulled 100,000 rows from 64uk-42ks...
Pulled 150,000 rows from 64uk-42ks...
Pulled 200,000 rows from 64uk-42ks...
Pulled 250,000 rows from 64uk-42ks...
Pulled 300,000 rows from 64uk-42ks...
Pulled 350,000 rows from 64uk-42ks...
Pulled 400,000 rows from 64uk-42ks...
Pulled 450,000 rows from 64uk-42ks...
Pulled 500,000 rows from 64uk-42ks...
Pulled 550,000 rows from 64uk-42ks...
Pulled 600,000 rows from 64uk-42ks...
Pulled 650,000 rows from 64uk-42ks...
Pulled 700,000 rows from 64uk-42ks...
Pulled 750,000 rows from 64uk-42ks...
Pulled 800,000 rows from 64uk-42ks...
Pulled 850,000 rows from 64uk-42ks...
Pulled 900,000 rows from 64uk-42ks...


In [9]:
print(pluto.shape)
print(pluto.columns.tolist())
display(pluto.head())

(858644, 19)
['borough', 'block', 'lot', 'bbl', 'cd', 'ct2010', 'cb2010', 'zipcode', 'landuse', 'bldgclass', 'lotarea', 'bldgarea', 'resarea', 'unitsres', 'unitstotal', 'yearbuilt', 'latitude', 'longitude', 'ownertype']


,borough,block,lot,bbl,cd,ct2010,cb2010,zipcode,landuse,bldgclass,lotarea,bldgarea,resarea,unitsres,unitstotal,yearbuilt,latitude,longitude,ownertype
0,QN,6173,23,4061730023.00000000,411,1123,1002,11361,1,A5,2660,1224,1224,1,1,1950,40.7690896,-73.7720738,NaN
1,QN,6173,24,4061730024.00000000,411,1123,1002,11361,1,A0,4200,1286,1286,1,1,1945,40.7688894,-73.7721503,NaN
2,QN,6169,26,4061690026.00000000,411,1123,2000,11361,1,B3,6000,2186,2186,2,2,1915,40.7679955,-73.7739873,NaN
3,QN,6172,1,4061720001.00000000,411,1123,2001,11361,2,C9,50500,27440,27440,36,36,1957,40.7671163,-73.7735210,NaN
4,QN,6172,6,4061720006.00000000,411,1123,2001,11361,2,C1,22050,27420,25920,36,36,1955,40.7675551,-73.7733751,NaN


In [10]:
pluto["bbl"] = (
    pd.to_numeric(pluto["bbl"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(10)
)

print(pluto["bbl"].str.len().value_counts(dropna=False))
display(pluto[["borough", "block", "lot", "bbl"]].head())

bbl
10    858644
Name: count, dtype: int64


,borough,block,lot,bbl
0,QN,6173,23,4061730023
1,QN,6173,24,4061730024
2,QN,6169,26,4061690026
3,QN,6172,1,4061720001
4,QN,6172,6,4061720006


3. Merge Lien Sale List and PLUTO

In [11]:
master_property = final_sale_property.merge(
    pluto,
    on="bbl",
    how="left",
    suffixes=("_lien", "_pluto")
)

master_property["matched_pluto"] = master_property["landuse"].notna().astype(int)

display(master_property["matched_pluto"].value_counts())
display(master_property["matched_pluto"].value_counts(normalize=True))

matched_pluto
1    7730
0    2033
Name: count, dtype: int64

matched_pluto
1    0.791765
0    0.208235
Name: proportion, dtype: float64

In [12]:
display(final_sale_property[["borough", "block", "lot", "bbl"]].head(10))
display(pluto[["borough", "block", "lot", "bbl"]].head(10))

print(final_sale_property["bbl"].str.len().value_counts(dropna=False))
print(pluto["bbl"].str.len().value_counts(dropna=False))

,borough,block,lot,bbl
65761,1,26,1260,1000261260
65762,1,26,1273,1000261273
65763,1,27,1040,1000271040
65764,1,31,1068,1000311068
65765,1,44,1265,1000441265
65766,1,53,1145,1000531145
65767,1,53,1153,1000531153
65768,1,53,1294,1000531294
65769,1,75,1013,1000751013
65770,1,76,1136,1000761136


,borough,block,lot,bbl
0,QN,6173,23,4061730023
1,QN,6173,24,4061730024
2,QN,6169,26,4061690026
3,QN,6172,1,4061720001
4,QN,6172,6,4061720006
5,QN,6172,19,4061720019
6,QN,6176,1,4061760001
7,BX,2603,83,2026030083
8,QN,6176,14,4061760014
9,BX,3648,43,2036480043


bbl
10    9763
Name: count, dtype: int64
bbl
10    858644
Name: count, dtype: int64


4. HPD Violations dataset

In [13]:
hpd = socrata_get(
    dataset_id="wvxf-dwi5",
    select="boro, block, lot, class, violationstatus",
    where="violationstatus='Open'"
)

Pulled 50,000 rows from wvxf-dwi5...
Pulled 100,000 rows from wvxf-dwi5...
Pulled 150,000 rows from wvxf-dwi5...
Pulled 200,000 rows from wvxf-dwi5...
Pulled 250,000 rows from wvxf-dwi5...
Pulled 300,000 rows from wvxf-dwi5...
Pulled 350,000 rows from wvxf-dwi5...
Pulled 400,000 rows from wvxf-dwi5...
Pulled 450,000 rows from wvxf-dwi5...
Pulled 500,000 rows from wvxf-dwi5...
Pulled 550,000 rows from wvxf-dwi5...
Pulled 600,000 rows from wvxf-dwi5...
Pulled 650,000 rows from wvxf-dwi5...
Pulled 700,000 rows from wvxf-dwi5...
Pulled 750,000 rows from wvxf-dwi5...
Pulled 800,000 rows from wvxf-dwi5...
Pulled 850,000 rows from wvxf-dwi5...
Pulled 900,000 rows from wvxf-dwi5...
Pulled 950,000 rows from wvxf-dwi5...
Pulled 1,000,000 rows from wvxf-dwi5...
Pulled 1,050,000 rows from wvxf-dwi5...
Pulled 1,100,000 rows from wvxf-dwi5...
Pulled 1,150,000 rows from wvxf-dwi5...
Pulled 1,200,000 rows from wvxf-dwi5...
Pulled 1,250,000 rows from wvxf-dwi5...
Pulled 1,300,000 rows from wvxf-dwi5...

In [14]:
# aggregates HPD Violation

# Clean columns
hpd.columns = (
    hpd.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# Create HPD BBL
boro_map = {
    "MANHATTAN": "1",
    "BRONX": "2",
    "BROOKLYN": "3",
    "QUEENS": "4",
    "STATEN ISLAND": "5"
}

hpd["boro_code"] = (
    hpd["boro"]
    .astype(str)
    .str.upper()
    .str.strip()
    .replace(boro_map)
)

hpd["bbl"] = make_bbl(hpd["boro_code"], hpd["block"], hpd["lot"])

# Aggregate to one row per property
hpd_summary = (
    hpd.assign(
        open_hpd_violation=1,
        open_b_or_c_violation=np.where(hpd["class"].isin(["B", "C"]), 1, 0),
        open_c_violation=np.where(hpd["class"].eq("C"), 1, 0)
    )
    .groupby("bbl", as_index=False)
    .agg(
        open_hpd_violations=("open_hpd_violation", "sum"),
        open_b_or_c_violations=("open_b_or_c_violation", "sum"),
        open_c_violations=("open_c_violation", "sum")
    )
)

print(hpd_summary.shape)
display(hpd_summary.head())

(156873, 4)


,bbl,open_hpd_violations,open_b_or_c_violations,open_c_violations
0,1000000000,86,57,2
1,1000050010,3,2,1
2,1000077501,36,23,11
3,1000110010,25,21,9
4,1000117501,1,0,0


5. Merge master bbl and HPD violations

In [15]:
master_property = master_property.merge(
    hpd_summary,
    on="bbl",
    how="left"
)

for col in ["open_hpd_violations", "open_b_or_c_violations", "open_c_violations"]:
    master_property[col] = master_property[col].fillna(0)

print(master_property.shape)
display(master_property[[
    "bbl",
    "tax_class_code",
    "building_class",
    "open_hpd_violations",
    "open_b_or_c_violations",
    "open_c_violations"
]].head())

(9763, 37)


,bbl,tax_class_code,building_class,open_hpd_violations,open_b_or_c_violations,open_c_violations
0,1000261260,2,R4,0.0,0.0,0.0
1,1000261273,2,R4,0.0,0.0,0.0
2,1000271040,2,R4,0.0,0.0,0.0
3,1000311068,2,R4,0.0,0.0,0.0
4,1000441265,2,R4,0.0,0.0,0.0


In [16]:
master_property[[
    "open_hpd_violations",
    "open_b_or_c_violations",
    "open_c_violations"
]].describe()

,open_hpd_violations,open_b_or_c_violations,open_c_violations
count,9763.000000,9763.000000,9763.000000
mean,14.689849,10.595821,3.043020
std,44.829899,36.002966,12.439906
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,8.000000,3.000000,0.000000
max,1564.000000,1328.000000,536.000000


6. Real Property Assessment Data (RPAD)

In [37]:
# Check available RPAD years using direct requests, because this needs $group
url = "https://data.cityofnewyork.us/resource/yjxr-fw8i.json"

params = {
    "$select": "`year`, count(*)",
    "$group": "`year`",
    "$order": "`year`"
}

r = requests.get(url, params=params)

print("Status:", r.status_code)
print(r.text[:500])

year_check = pd.DataFrame(r.json())
display(year_check)

Status: 200
[{"year":"2010/11","count":"1070995"}
,{"year":"2011/12","count":"1080569"}
,{"year":"2012/13","count":"1086396"}
,{"year":"2013/14","count":"1088350"}
,{"year":"2014/15","count":"1093327"}
,{"year":"2015/16","count":"1096247"}
,{"year":"2016/17","count":"1103323"}
,{"year":"2017/18","count":"1110058"}
,{"year":"2018/19","count":"1116592"}]



,year,count
0,2010/11,1070995
1,2011/12,1080569
2,2012/13,1086396
3,2013/14,1088350
4,2014/15,1093327
5,2015/16,1096247
6,2016/17,1103323
7,2017/18,1110058
8,2018/19,1116592


In [38]:
display(year_check)

,year,count
0,2010/11,1070995
1,2011/12,1080569
2,2012/13,1086396
3,2013/14,1088350
4,2014/15,1093327
5,2015/16,1096247
6,2016/17,1103323
7,2017/18,1110058
8,2018/19,1116592


In [39]:
dof_latest = socrata_get(
    dataset_id="yjxr-fw8i",
    select=", ".join(dof_cols),
    where="`year`='2018/19' AND period='FINAL'",
    limit=50000
)

print(dof_latest.shape)
display(dof_latest.head())

Pulled 50,000 rows from yjxr-fw8i...
Pulled 100,000 rows from yjxr-fw8i...
Pulled 150,000 rows from yjxr-fw8i...
Pulled 200,000 rows from yjxr-fw8i...
Pulled 250,000 rows from yjxr-fw8i...
Pulled 300,000 rows from yjxr-fw8i...
Pulled 350,000 rows from yjxr-fw8i...
Pulled 400,000 rows from yjxr-fw8i...
Pulled 450,000 rows from yjxr-fw8i...
Pulled 500,000 rows from yjxr-fw8i...
Pulled 550,000 rows from yjxr-fw8i...
Pulled 600,000 rows from yjxr-fw8i...
Pulled 650,000 rows from yjxr-fw8i...
Pulled 700,000 rows from yjxr-fw8i...
Pulled 750,000 rows from yjxr-fw8i...
Pulled 800,000 rows from yjxr-fw8i...
Pulled 850,000 rows from yjxr-fw8i...
Pulled 900,000 rows from yjxr-fw8i...
Pulled 950,000 rows from yjxr-fw8i...
Pulled 1,000,000 rows from yjxr-fw8i...
Pulled 1,050,000 rows from yjxr-fw8i...
Pulled 1,100,000 rows from yjxr-fw8i...
Pulled 1,150,000 rows from yjxr-fw8i...
(1116592, 16)


,bble,boro,block,lot,owner,bldgcl,taxclass,fullval,avland,avtot,exland,extot,period,year,valtype,zip
0,1000163859,1,16,3859,"CHEN, QI TOM",R4,2,354180,3310,159381,3310,159381,FINAL,2018/19,AC-TR,NaN
1,1000730028,1,73,28,NYC DSBS,V1,4,3515000,1581750,1581750,1581750,1581750,FINAL,2018/19,AC-TR,10038
2,1000730029,1,73,29,NYC DSBS,Y7,4,8215000,2812050,3696750,2812050,3696750,FINAL,2018/19,AC-TR,10038
3,1000297504,1,29,7504,NaN,R0,2,0,0,0,0,0,FINAL,2018/19,AC-TR,10004
4,1000360012,1,36,12,NYC DSBS,Y7,4,26246000,9180000,11810700,9180000,11810700,FINAL,2018/19,AC-TR,NaN


In [40]:
# ------------------------------------------------------------
# Clean DOF valuation data
# ------------------------------------------------------------

dof_latest.columns = (
    dof_latest.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# Keep only standard 10-digit BBLs
dof_latest = dof_latest[
    dof_latest["bble"].astype(str).str.match(r"^\d{10}$")
].copy()

# Create clean BBL
dof_latest["bbl"] = (
    dof_latest["bble"]
    .astype(str)
    .str.zfill(10)
)

# Convert value fields
value_cols = ["fullval", "avland", "avtot", "exland", "extot"]

for col in value_cols:
    dof_latest[col] = pd.to_numeric(dof_latest[col], errors="coerce")

# ------------------------------------------------------------
# Deduplicate to one row per property
# ------------------------------------------------------------

dof_property = (
    dof_latest
    .sort_values(["bbl"])
    .drop_duplicates("bbl")
    .copy()
)

print("DOF property-level shape:", dof_property.shape)

display(dof_property[[
    "bbl",
    "taxclass",
    "fullval",
    "avtot",
    "extot"
]].head())

DOF property-level shape: (1112066, 17)


,bbl,taxclass,fullval,avtot,extot
3883,1000010010,4,368983000,166042350,166042350
7638,1000010101,4,30166000,13574700,13574700
11251,1000010201,4,255678000,115055100,115055100
14779,1000010301,4,0,0,0
19329,1000010401,4,0,0,0


7. Merge master and RPAD

In [41]:
# ------------------------------------------------------------
# Merge DOF data into master_property
# ------------------------------------------------------------

master_property = master_property.merge(
    dof_property,
    on="bbl",
    how="left",
    suffixes=("", "_dof")
)

# Check match quality
master_property["matched_dof"] = (
    master_property["fullval"].notna()
).astype(int)

print(master_property.shape)

display(master_property["matched_dof"].value_counts())
display(master_property["matched_dof"].value_counts(normalize=True))

(9763, 54)


matched_dof
1    9571
0     192
Name: count, dtype: int64

matched_dof
1    0.980334
0    0.019666
Name: proportion, dtype: float64

8. ACS Data

In [42]:
import requests
import pandas as pd
import numpy as np

ACS_YEAR = 2023
ACS_BASE = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

nyc_counties = {
    "061": "Manhattan",
    "005": "Bronx",
    "047": "Brooklyn",
    "081": "Queens",
    "085": "Staten Island"
}

acs_vars = {
    "B19013_001E": "median_household_income",
    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",
    "B25003_001E": "occupied_units",
    "B25003_002E": "owner_occupied_units",
    "B25002_001E": "housing_units",
    "B25002_003E": "vacant_units",
    "B25070_001E": "renter_cost_burden_universe",
    "B25070_007E": "rent_30_34",
    "B25070_008E": "rent_35_39",
    "B25070_009E": "rent_40_49",
    "B25070_010E": "rent_50_plus",
    "B03002_001E": "race_eth_total",
    "B03002_003E": "white_nonhispanic",
    "B03002_004E": "black_nonhispanic",
    "B03002_006E": "asian_nonhispanic",
    "B03002_012E": "hispanic",
    "B01001_001E": "age_total",
    "B15003_001E": "education_total",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional_degree",
    "B15003_025E": "doctorate"
}

age_65_vars = [
    "B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E",
    "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E"
]

all_vars = list(acs_vars.keys()) + age_65_vars

def get_acs_county(county_code):
    params = {
        "get": "NAME," + ",".join(all_vars),
        "for": "tract:*",
        "in": f"state:36 county:{county_code}"
    }
    
    r = requests.get(ACS_BASE, params=params)
    r.raise_for_status()
    
    data = r.json()
    return pd.DataFrame(data[1:], columns=data[0])

acs = pd.concat(
    [get_acs_county(c) for c in nyc_counties.keys()],
    ignore_index=True
)

acs["geoid"] = acs["state"] + acs["county"] + acs["tract"]

acs = acs.rename(columns=acs_vars)

for col in acs.columns:
    if col not in ["NAME", "state", "county", "tract", "geoid"]:
        acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs["poverty_rate"] = acs["poverty_count"] / acs["poverty_universe"]
acs["homeownership_rate"] = acs["owner_occupied_units"] / acs["occupied_units"]
acs["vacancy_rate"] = acs["vacant_units"] / acs["housing_units"]

acs["rent_burdened_count"] = (
    acs["rent_30_34"] +
    acs["rent_35_39"] +
    acs["rent_40_49"] +
    acs["rent_50_plus"]
)

acs["rent_burden_rate"] = acs["rent_burdened_count"] / acs["renter_cost_burden_universe"]
acs["severe_rent_burden_rate"] = acs["rent_50_plus"] / acs["renter_cost_burden_universe"]

acs["black_rate"] = acs["black_nonhispanic"] / acs["race_eth_total"]
acs["hispanic_rate"] = acs["hispanic"] / acs["race_eth_total"]
acs["asian_rate"] = acs["asian_nonhispanic"] / acs["race_eth_total"]
acs["white_nonhispanic_rate"] = acs["white_nonhispanic"] / acs["race_eth_total"]

acs["age_65_plus_count"] = acs[age_65_vars].sum(axis=1)
acs["age_65_plus_rate"] = acs["age_65_plus_count"] / acs["age_total"]

acs["bachelors_plus_count"] = (
    acs["bachelors"] +
    acs["masters"] +
    acs["professional_degree"] +
    acs["doctorate"]
)

acs["bachelors_plus_rate"] = acs["bachelors_plus_count"] / acs["education_total"]

acs_clean = acs[[
    "geoid", "NAME",
    "median_household_income",
    "poverty_rate",
    "homeownership_rate",
    "vacancy_rate",
    "rent_burden_rate",
    "severe_rent_burden_rate",
    "black_rate",
    "hispanic_rate",
    "asian_rate",
    "white_nonhispanic_rate",
    "age_65_plus_rate",
    "bachelors_plus_rate"
]].copy()

print(acs_clean.shape)
display(acs_clean.head())

(2327, 14)


,geoid,NAME,median_household_income,poverty_rate,homeownership_rate,vacancy_rate,rent_burden_rate,severe_rent_burden_rate,black_rate,hispanic_rate,asian_rate,white_nonhispanic_rate,age_65_plus_rate,bachelors_plus_rate
0,36061000100,Census Tract 1; New York County; New York,-666666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36061000201,Census Tract 2.01; New York County; New York,38657,0.530345,0.000000,0.113147,0.596598,0.217497,0.139798,0.579050,0.189252,0.066589,0.147586,0.157380
2,36061000202,Census Tract 2.02; New York County; New York,40182,0.249493,0.294892,0.091918,0.503556,0.234667,0.145948,0.361408,0.225844,0.252091,0.225699,0.366179
3,36061000500,Census Tract 5; New York County; New York,-666666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36061000600,Census Tract 6; New York County; New York,34473,0.366660,0.052642,0.100686,0.524059,0.253674,0.066902,0.292563,0.493467,0.111504,0.264957,0.253202
